# Notebook 4 — Deep Learning: LSTM & BERT Sentiment Classification
### Bidirectional LSTM · DistilBERT Fine-Tuning

| Item | Detail |
|------|--------|
| **Input** | `Dataset/results/gold/gold_reviews.parquet` |
| **Task** | Binary sentiment classification (Negative=0 / Positive=1) |
| **LSTM sample** | 200k balanced reviews (100k per class) |
| **BERT sample** | 100k balanced reviews (50k per class) |
| **Output** | Trained models + metrics saved to `Dataset/results/models/` |

> **⚠️ Data Leakage Fix:** The classical ML notebook achieved perfect 1.0 scores because
> `user_star_deviation` and `biz_star_deviation` directly encode `stars` (the label source).
> These features are **excluded** here. LSTM and BERT operate on raw review **text** only.

## 1 — Install Dependencies

In [ ]:
!pip install transformers datasets torch torchmetrics pyarrow matplotlib seaborn scikit-learn -q
!pip install tensorflow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 55.8 MB/s eta 0:00:00


## 2 — Imports

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import pyarrow.parquet as pq

# TensorFlow / Keras (LSTM)
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# HuggingFace (BERT)
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)

# Sklearn metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve
)

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

# GPU check
print('TensorFlow version :', tf.__version__)
print('PyTorch version    :', torch.__version__)
print('GPU (TF)           :', tf.config.list_physical_devices('GPU'))
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch device     : {DEVICE}')

TensorFlow version : 2.20.0
PyTorch version    : 2.10.0+cu128
GPU (TF)           : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
PyTorch device     : cuda


## 3 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 4 — Paths & Directories

In [ ]:
RESULTS_DIR = '/content/drive/MyDrive/Dataset/results'
GOLD_DIR    = f'{RESULTS_DIR}/gold'
MODELS_DIR  = f'{RESULTS_DIR}/models'
PLOTS_DIR   = f'{RESULTS_DIR}/dl_plots'

for d in [MODELS_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

def save_fig(name):
    path = os.path.join(PLOTS_DIR, f'{name}.png')
    plt.savefig(path, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'  Saved → {path}')

print('Directories ready.')

Directories ready.


---
## 5 — Load & Prepare Data
> Read gold parquet with **pyarrow** directly (no Spark needed — we work on a balanced sample).

In [ ]:
# Read only columns we need
gold_pq = pq.read_table(
    f'{GOLD_DIR}/gold_reviews.parquet',
    columns=['text', 'sentiment_binary', 'stars',
             'char_count', 'word_count', 'review_year']
).to_pandas()

# Drop neutral (3★)
gold_pq = gold_pq.dropna(subset=['sentiment_binary'])
gold_pq['label'] = gold_pq['sentiment_binary'].astype(int)
gold_pq = gold_pq.dropna(subset=['text'])
gold_pq['text'] = gold_pq['text'].astype(str).str.strip()
gold_pq = gold_pq[gold_pq['text'].str.len() > 10]  # remove near-empty reviews

print(f'Total usable reviews: {len(gold_pq):,}')
print(gold_pq['label'].value_counts().rename({0:'Negative', 1:'Positive'}).to_string())

Total usable reviews: 6,298,179
label
Positive    4684444
Negative    1613735


In [ ]:
# --- Balanced sample for LSTM (200k total) ---
LSTM_PER_CLASS = 100_000
neg_pool = gold_pq[gold_pq['label'] == 0]
pos_pool = gold_pq[gold_pq['label'] == 1]

lstm_df = pd.concat([
    neg_pool.sample(n=LSTM_PER_CLASS, random_state=42),
    pos_pool.sample(n=LSTM_PER_CLASS, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

# --- Balanced sample for BERT (100k total) ---
BERT_PER_CLASS = 50_000
bert_df = pd.concat([
    neg_pool.sample(n=BERT_PER_CLASS, random_state=99),
    pos_pool.sample(n=BERT_PER_CLASS, random_state=99)
]).sample(frac=1, random_state=99).reset_index(drop=True)

print(f'LSTM sample : {len(lstm_df):,} rows  |  {lstm_df["label"].value_counts().to_dict()}')
print(f'BERT sample : {len(bert_df):,} rows  |  {bert_df["label"].value_counts().to_dict()}')

LSTM sample : 200,000 rows  |  {1: 100000, 0: 100000}
BERT sample : 100,000 rows  |  {1: 50000, 0: 50000}


---
# Part 1 — LSTM (Bidirectional)

## 6 — LSTM: Text Preprocessing

In [ ]:
VOCAB_SIZE = 50_000
MAX_LEN    = 200      # tokens per review
OOV_TOKEN  = '<OOV>'

# Train/val/test split (70/15/15)
X_lstm = lstm_df['text'].values
y_lstm = lstm_df['label'].values

X_tr, X_tmp, y_tr, y_tmp = train_test_split(X_lstm, y_lstm, test_size=0.30, random_state=42, stratify=y_lstm)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=42, stratify=y_tmp)

# Fit tokenizer on train only
keras_tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOKEN)
keras_tokenizer.fit_on_texts(X_tr)

def encode(texts):
    seqs = keras_tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LEN, padding='post', truncating='post')

X_tr_enc  = encode(X_tr)
X_val_enc = encode(X_val)
X_te_enc  = encode(X_te)

print(f'Vocab size (fitted) : {len(keras_tokenizer.word_index):,}')
print(f'Train   : {X_tr_enc.shape}   Labels: {np.bincount(y_tr)}')
print(f'Val     : {X_val_enc.shape}  Labels: {np.bincount(y_val)}')
print(f'Test    : {X_te_enc.shape}   Labels: {np.bincount(y_te)}')

Vocab size (fitted) : 94,430
Train   : (140000, 200)   Labels: [70000 70000]
Val     : (30000, 200)  Labels: [15000 15000]
Test    : (30000, 200)   Labels: [15000 15000]


## 7 — LSTM: Model Architecture

In [ ]:
EMBED_DIM  = 128
LSTM_UNITS = 128

inp = Input(shape=(MAX_LEN,), name='text_input')
x   = layers.Embedding(VOCAB_SIZE, EMBED_DIM, mask_zero=True, name='embedding')(inp)
x   = layers.SpatialDropout1D(0.2)(x)
x   = layers.Bidirectional(
          layers.LSTM(LSTM_UNITS, return_sequences=True, dropout=0.2, recurrent_dropout=0.1),
          name='bilstm_1'
      )(x)
x   = layers.Bidirectional(
          layers.LSTM(64, dropout=0.2),
          name='bilstm_2'
      )(x)
x   = layers.Dense(64, activation='relu', name='dense_1')(x)
x   = layers.Dropout(0.3)(x)
out = layers.Dense(1, activation='sigmoid', name='output')(x)

lstm_model = Model(inputs=inp, outputs=out)
lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)
lstm_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text_input          │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 200, 128)  │  6,400,000 │ text_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d   │ (None, 200, 128)  │          0 │ embedding[0][0]   │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 200)       │          0 │ text_input[0][0]  │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_1            │ (None, 200, 256)  │    263,168 │ spatial_dropout1… │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_2            │ (None, 128)       │    164,352 │ bilstm_1[0][0],   │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      8,256 │ bilstm_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 1)         │         65 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,835,841 (26.08 MB)

 Trainable params: 6,835,841 (26.08 MB)

 Non-trainable params: 0 (0.00 B)

## 8 — LSTM: Train

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_auc', patience=3, restore_best_weights=True, mode='max', verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)
]

print('Training BiLSTM ...')
lstm_history = lstm_model.fit(
    X_tr_enc, y_tr,
    validation_data=(X_val_enc, y_val),
    epochs=5,
    batch_size=512,
    class_weight={0: 1.0, 1: 1.0},  # already balanced
    callbacks=callbacks,
    verbose=1
)

Training BiLSTM ...
Epoch 1/5
274/274 ━━━━━━━━━━━━━━━━━━━━ 313s 1s/step - accuracy: 0.9841 - auc: 0.9978 - loss: 0.0480 - val_accuracy: 0.9511 - val_auc: 0.9873 - val_loss: 0.1529 - learning_rate: 5.0000e-04
Epoch 2/5
274/274 ━━━━━━━━━━━━━━━━━━━━ 312s 1s/step - accuracy: 0.9878 - auc: 0.9985 - loss: 0.0374 - val_accuracy: 0.9519 - val_auc: 0.9836 - val_loss: 0.1803 - learning_rate: 5.0000e-04
Epoch 3/5
274/274 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9902 - auc: 0.9991 - loss: 0.0286
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
274/274 ━━━━━━━━━━━━━━━━━━━━ 312s 1s/step - accuracy: 0.9897 - auc: 0.9990 - loss: 0.0306 - val_accuracy: 0.9509 - val_auc: 0.9839 - val_loss: 0.1812 - learning_rate: 5.0000e-04
Epoch 4/5
274/274 ━━━━━━━━━━━━━━━━━━━━ 313s 1s/step - accuracy: 0.9935 - auc: 0.9994 - loss: 0.0205 - val_accuracy: 0.9510 - val_auc: 0.9820 - val_loss: 0.1975 - learning_rate: 2.5000e-04
Epoch 4: early stopping
Restoring model weights from the end of 

## 9 — LSTM: Training Curves

In [ ]:
hist = lstm_history.history
epochs = range(1, len(hist['loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('BiLSTM Training Curves', fontsize=14, fontweight='bold')

for ax, (train_key, val_key, title) in zip(axes, [
    ('loss',     'val_loss',     'Loss'),
    ('accuracy', 'val_accuracy', 'Accuracy'),
    ('auc',      'val_auc',      'AUC-ROC'),
]):
    ax.plot(epochs, hist[train_key], 'b-o', label='Train', linewidth=2)
    ax.plot(epochs, hist[val_key],   'r-o', label='Val',   linewidth=2)
    ax.set_title(title); ax.set_xlabel('Epoch')
    ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
save_fig('01_lstm_training_curves')

  Saved → /content/drive/MyDrive/Dataset/results/dl_plots/01_lstm_training_curves.png


## 10 — LSTM: Evaluate on Test Set

In [ ]:
lstm_proba = lstm_model.predict(X_te_enc, batch_size=512, verbose=0).flatten()
lstm_preds = (lstm_proba >= 0.5).astype(int)

lstm_auc = roc_auc_score(y_te, lstm_proba)
lstm_acc = accuracy_score(y_te, lstm_preds)
lstm_rpt = classification_report(y_te, lstm_preds, target_names=['Negative','Positive'], output_dict=True)

print('=== BiLSTM Test Results ===')
print(f'  AUC-ROC   : {lstm_auc:.4f}')
print(f'  Accuracy  : {lstm_acc:.4f}')
print(f'  F1 (wtd)  : {lstm_rpt["weighted avg"]["f1-score"]:.4f}')
print(f'  Precision : {lstm_rpt["weighted avg"]["precision"]:.4f}')
print(f'  Recall    : {lstm_rpt["weighted avg"]["recall"]:.4f}')
print(f'\nClassification Report:')
print(classification_report(y_te, lstm_preds, target_names=['Negative','Positive']))

=== BiLSTM Test Results ===
  AUC-ROC   : 0.9891
  Accuracy  : 0.9491
  F1 (wtd)  : 0.9491
  Precision : 0.9493
  Recall    : 0.9491

Classification Report:
              precision    recall  f1-score   support

    Negative       0.96      0.94      0.95     15000
    Positive       0.94      0.96      0.95     15000

    accuracy                           0.95     30000
   macro avg       0.95      0.95      0.95     30000
weighted avg       0.95      0.95      0.95     30000



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('BiLSTM — Evaluation', fontsize=14, fontweight='bold')

# Confusion Matrix
cm = confusion_matrix(y_te, lstm_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=['Negative','Positive'],
            yticklabels=['Negative','Positive'], ax=axes[0])
for i in range(2):
    for j in range(2):
        axes[0].text(j+0.5, i+0.72, f'({cm[i,j]:,})', ha='center', fontsize=8, color='gray')
axes[0].set_title('Confusion Matrix'); axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

# ROC Curve
fpr, tpr, _ = roc_curve(y_te, lstm_proba)
axes[1].plot(fpr, tpr, color='steelblue', linewidth=2, label=f'AUC = {lstm_auc:.4f}')
axes[1].plot([0,1],[0,1],'k--', linewidth=1, label='Random')
axes[1].set_title('ROC Curve'); axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
save_fig('02_lstm_evaluation')

  Saved → /content/drive/MyDrive/Dataset/results/dl_plots/02_lstm_evaluation.png


## 11 — Save LSTM Model

In [ ]:
lstm_model.save(f'{MODELS_DIR}/lstm_sentiment_model.h5')

# Save tokenizer config
import json
tokenizer_json = keras_tokenizer.to_json()
with open(f'{MODELS_DIR}/lstm_tokenizer.json', 'w') as f:
    f.write(tokenizer_json)

# Save LSTM metrics
lstm_metrics = {
    'model': 'BiLSTM',
    'sample_size': len(lstm_df),
    'auc_roc': round(lstm_auc, 4),
    'accuracy': round(lstm_acc, 4),
    'f1_weighted': round(lstm_rpt['weighted avg']['f1-score'], 4),
    'precision_weighted': round(lstm_rpt['weighted avg']['precision'], 4),
    'recall_weighted': round(lstm_rpt['weighted avg']['recall'], 4),
}
with open(f'{RESULTS_DIR}/lstm_metrics.json', 'w') as f:
    json.dump(lstm_metrics, f, indent=2)

print('LSTM model saved.')
print(json.dumps(lstm_metrics, indent=2))

LSTM model saved.
{
  "model": "BiLSTM",
  "sample_size": 200000,
  "auc_roc": 0.9891,
  "accuracy": 0.9491,
  "f1_weighted": 0.9491,
  "precision_weighted": 0.9493,
  "recall_weighted": 0.9491
}


---
# Part 2 — BERT (DistilBERT Fine-Tuning)

## 12 — BERT: Tokenization & Dataset

In [ ]:
BERT_MODEL_NAME = 'distilbert-base-uncased'
BERT_MAX_LEN    = 128
BERT_BATCH_SIZE = 32
BERT_EPOCHS     = 3
BERT_LR         = 2e-5

bert_tokenizer = DistilBertTokenizerFast.from_pretrained(BERT_MODEL_NAME)

# Train/val/test split (70/15/15)
X_bert = bert_df['text'].values
y_bert = bert_df['label'].values

X_btr, X_btmp, y_btr, y_btmp = train_test_split(X_bert, y_bert, test_size=0.30, random_state=42, stratify=y_bert)
X_bval, X_bte, y_bval, y_bte = train_test_split(X_btmp, y_btmp, test_size=0.50, random_state=42, stratify=y_btmp)

print(f'BERT Train : {len(X_btr):,}  Val : {len(X_bval):,}  Test : {len(X_bte):,}')
print(f'Tokenizer  : {BERT_MODEL_NAME}')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

BERT Train : 70,000  Val : 15,000  Test : 15,000
Tokenizer  : distilbert-base-uncased


In [ ]:
class YelpReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_ds = YelpReviewDataset(X_btr,  y_btr,  bert_tokenizer, BERT_MAX_LEN)
val_ds   = YelpReviewDataset(X_bval, y_bval, bert_tokenizer, BERT_MAX_LEN)
test_ds  = YelpReviewDataset(X_bte,  y_bte,  bert_tokenizer, BERT_MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BERT_BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BERT_BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BERT_BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches : {len(train_loader):,}')
print(f'Val batches   : {len(val_loader):,}')
print(f'Test batches  : {len(test_loader):,}')

Train batches : 2,188
Val batches   : 469
Test batches  : 469


## 13 — BERT: Model & Optimizer

In [ ]:
bert_model = DistilBertForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME,
    num_labels=2
).to(DEVICE)

optimizer = torch.optim.AdamW(
    bert_model.parameters(),
    lr=BERT_LR,
    weight_decay=0.01
)

total_steps = len(train_loader) * BERT_EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.06),
    num_training_steps=total_steps
)

print(f'Model parameters : {sum(p.numel() for p in bert_model.parameters()):,}')
print(f'Total train steps: {total_steps:,}')

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model parameters : 66,955,010
Total train steps: 6,564


## 14 — BERT: Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in loader:
        input_ids  = batch['input_ids'].to(device)
        attn_mask  = batch['attention_mask'].to(device)
        labels     = batch['label'].to(device)
        optimizer.zero_grad()
        outputs    = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
        loss       = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        preds = outputs.logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += len(labels)
    return total_loss / len(loader), correct / total


def eval_epoch(model, loader, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_proba, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids  = batch['input_ids'].to(device)
            attn_mask  = batch['attention_mask'].to(device)
            labels     = batch['label'].to(device)
            outputs    = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
            total_loss += outputs.loss.item()
            proba = torch.softmax(outputs.logits, dim=1)[:, 1].cpu().numpy()
            preds = outputs.logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += len(labels)
            all_proba.extend(proba)
            all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), correct / total, np.array(all_proba), np.array(all_labels)


bert_train_history = {'loss': [], 'acc': [], 'val_loss': [], 'val_acc': [], 'val_auc': []}
best_val_auc = 0

for epoch in range(1, BERT_EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(bert_model, train_loader, optimizer, scheduler, DEVICE)
    vl_loss, vl_acc, vl_proba, vl_labels = eval_epoch(bert_model, val_loader, DEVICE)
    vl_auc = roc_auc_score(vl_labels, vl_proba)

    bert_train_history['loss'].append(tr_loss)
    bert_train_history['acc'].append(tr_acc)
    bert_train_history['val_loss'].append(vl_loss)
    bert_train_history['val_acc'].append(vl_acc)
    bert_train_history['val_auc'].append(vl_auc)

    print(f'Epoch {epoch}/{BERT_EPOCHS}  '
          f'Train Loss={tr_loss:.4f} Acc={tr_acc:.4f}  '
          f'Val Loss={vl_loss:.4f} Acc={vl_acc:.4f} AUC={vl_auc:.4f}')

    if vl_auc > best_val_auc:
        best_val_auc = vl_auc
        bert_model.save_pretrained(f'{MODELS_DIR}/bert_best_checkpoint')
        bert_tokenizer.save_pretrained(f'{MODELS_DIR}/bert_best_checkpoint')
        print(f'  ✓ New best checkpoint saved (AUC={vl_auc:.4f})')

Epoch 1/3  Train Loss=0.1681 Acc=0.9344  Val Loss=0.1132 Acc=0.9609 AUC=0.9924


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ New best checkpoint saved (AUC=0.9924)
Epoch 2/3  Train Loss=0.0796 Acc=0.9736  Val Loss=0.1147 Acc=0.9651 AUC=0.9937


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ New best checkpoint saved (AUC=0.9937)
Epoch 3/3  Train Loss=0.0432 Acc=0.9879  Val Loss=0.1447 Acc=0.9643 AUC=0.9934


## 15 — BERT: Training Curves

In [ ]:
epochs_r = range(1, BERT_EPOCHS + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('DistilBERT Fine-Tuning Curves', fontsize=14, fontweight='bold')

for ax, (tr_key, vl_key, title) in zip(axes, [
    ('loss',     'val_loss', 'Loss'),
    ('acc',      'val_acc',  'Accuracy'),
    ('val_auc',  None,       'Validation AUC-ROC'),
]):
    ax.plot(epochs_r, bert_train_history[tr_key], 'b-o', label='Train', linewidth=2)
    if vl_key:
        ax.plot(epochs_r, bert_train_history[vl_key], 'r-o', label='Val', linewidth=2)
        ax.legend()
    ax.set_title(title); ax.set_xlabel('Epoch')
    ax.grid(alpha=0.3)

plt.tight_layout()
save_fig('03_bert_training_curves')

  Saved → /content/drive/MyDrive/Dataset/results/dl_plots/03_bert_training_curves.png


## 16 — BERT: Evaluate on Test Set

In [ ]:
# Load best checkpoint
bert_best = DistilBertForSequenceClassification.from_pretrained(
    f'{MODELS_DIR}/bert_best_checkpoint'
).to(DEVICE)

_, bert_acc, bert_proba, bert_true = eval_epoch(bert_best, test_loader, DEVICE)
bert_preds = (bert_proba >= 0.5).astype(int)
bert_auc   = roc_auc_score(bert_true, bert_proba)
bert_rpt   = classification_report(bert_true, bert_preds,
                                    target_names=['Negative','Positive'],
                                    output_dict=True)

print('=== DistilBERT Test Results ===')
print(f'  AUC-ROC   : {bert_auc:.4f}')
print(f'  Accuracy  : {bert_acc:.4f}')
print(f'  F1 (wtd)  : {bert_rpt["weighted avg"]["f1-score"]:.4f}')
print(f'  Precision : {bert_rpt["weighted avg"]["precision"]:.4f}')
print(f'  Recall    : {bert_rpt["weighted avg"]["recall"]:.4f}')
print(f'\nClassification Report:')
print(classification_report(bert_true, bert_preds, target_names=['Negative','Positive']))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

=== DistilBERT Test Results ===
  AUC-ROC   : 0.9931
  Accuracy  : 0.9624
  F1 (wtd)  : 0.9624
  Precision : 0.9624
  Recall    : 0.9624

Classification Report:
              precision    recall  f1-score   support

    Negative       0.96      0.96      0.96      7500
    Positive       0.96      0.96      0.96      7500

    accuracy                           0.96     15000
   macro avg       0.96      0.96      0.96     15000
weighted avg       0.96      0.96      0.96     15000



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('DistilBERT — Evaluation', fontsize=14, fontweight='bold')

# Confusion matrix
cm_bert = confusion_matrix(bert_true, bert_preds)
cm_norm = cm_bert.astype('float') / cm_bert.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Purples',
            xticklabels=['Negative','Positive'],
            yticklabels=['Negative','Positive'], ax=axes[0])
for i in range(2):
    for j in range(2):
        axes[0].text(j+0.5, i+0.72, f'({cm_bert[i,j]:,})', ha='center', fontsize=8, color='gray')
axes[0].set_title('Confusion Matrix'); axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

# ROC curve
fpr_bert, tpr_bert, _ = roc_curve(bert_true, bert_proba)
axes[1].plot(fpr_bert, tpr_bert, color='purple', linewidth=2, label=f'AUC = {bert_auc:.4f}')
axes[1].plot([0,1],[0,1],'k--', linewidth=1, label='Random')
axes[1].set_title('ROC Curve'); axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
save_fig('04_bert_evaluation')

  Saved → /content/drive/MyDrive/Dataset/results/dl_plots/04_bert_evaluation.png


## 17 — Save BERT Metrics

In [ ]:
bert_metrics = {
    'model': 'DistilBERT',
    'sample_size': len(bert_df),
    'auc_roc': round(bert_auc, 4),
    'accuracy': round(float(bert_acc), 4),
    'f1_weighted': round(bert_rpt['weighted avg']['f1-score'], 4),
    'precision_weighted': round(bert_rpt['weighted avg']['precision'], 4),
    'recall_weighted': round(bert_rpt['weighted avg']['recall'], 4),
}
with open(f'{RESULTS_DIR}/bert_metrics.json', 'w') as f:
    json.dump(bert_metrics, f, indent=2)

print('BERT metrics saved.')
print(json.dumps(bert_metrics, indent=2))

BERT metrics saved.
{
  "model": "DistilBERT",
  "sample_size": 100000,
  "auc_roc": 0.9931,
  "accuracy": 0.9624,
  "f1_weighted": 0.9624,
  "precision_weighted": 0.9624,
  "recall_weighted": 0.9624
}


---
# Part 3 — Model Comparison

## 18 — LSTM vs BERT vs Classical ML

In [ ]:
# Load classical ML metrics from previous notebook
try:
    classical_df = pd.read_csv(f'{RESULTS_DIR}/ml_metrics.csv')
    # Note: classical ML scores were 1.0 due to data leakage — mark as such
    classical_df['Note'] = 'Data leakage (star-derived features)'
except FileNotFoundError:
    classical_df = pd.DataFrame()

dl_metrics = pd.DataFrame([
    {'Model': 'BiLSTM (200k, text only)',
     'AUC-ROC': lstm_metrics['auc_roc'],
     'Accuracy': lstm_metrics['accuracy'],
     'F1': lstm_metrics['f1_weighted'],
     'Precision': lstm_metrics['precision_weighted'],
     'Recall': lstm_metrics['recall_weighted'],
     'Note': 'Text only, balanced'},
    {'Model': 'DistilBERT fine-tuned (100k, text only)',
     'AUC-ROC': bert_metrics['auc_roc'],
     'Accuracy': bert_metrics['accuracy'],
     'F1': bert_metrics['f1_weighted'],
     'Precision': bert_metrics['precision_weighted'],
     'Recall': bert_metrics['recall_weighted'],
     'Note': 'Text only, balanced'},
])

print('=== Deep Learning Models ===')
display(dl_metrics[['Model','AUC-ROC','Accuracy','F1','Precision','Recall']].to_string(index=False))

# Save combined metrics
dl_metrics.to_csv(f'{RESULTS_DIR}/dl_metrics.csv', index=False)
print('\nDL metrics saved.')

=== Deep Learning Models ===


'                                  Model  AUC-ROC  Accuracy     F1  Precision  Recall\n               BiLSTM (200k, text only)   0.9891    0.9491 0.9491     0.9493  0.9491\nDistilBERT fine-tuned (100k, text only)   0.9931    0.9624 0.9624     0.9624  0.9624'


DL metrics saved.


In [ ]:
# Side-by-side ROC curves: LSTM vs BERT
plt.figure(figsize=(9, 7))

# LSTM
fpr_lstm, tpr_lstm, _ = roc_curve(y_te, lstm_proba)
plt.plot(fpr_lstm, tpr_lstm, color='steelblue', linewidth=2,
         label=f'BiLSTM (AUC={lstm_auc:.4f})')

# BERT
plt.plot(fpr_bert, tpr_bert, color='purple', linewidth=2,
         label=f'DistilBERT (AUC={bert_auc:.4f})')

plt.plot([0,1],[0,1],'k--', linewidth=1, label='Random Baseline')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — LSTM vs DistilBERT', fontsize=14, fontweight='bold')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
save_fig('05_lstm_vs_bert_roc')

  Saved → /content/drive/MyDrive/Dataset/results/dl_plots/05_lstm_vs_bert_roc.png


In [ ]:
# Bar chart: LSTM vs BERT metrics
metrics_to_plot = ['AUC-ROC', 'Accuracy', 'F1', 'Precision', 'Recall']
x = np.arange(len(metrics_to_plot))
width = 0.30

fig, ax = plt.subplots(figsize=(12, 6))
fig.suptitle('BiLSTM vs DistilBERT — Metric Comparison', fontsize=14, fontweight='bold')

lstm_vals = [lstm_metrics['auc_roc'], lstm_metrics['accuracy'],
             lstm_metrics['f1_weighted'], lstm_metrics['precision_weighted'],
             lstm_metrics['recall_weighted']]
bert_vals = [bert_metrics['auc_roc'], bert_metrics['accuracy'],
             bert_metrics['f1_weighted'], bert_metrics['precision_weighted'],
             bert_metrics['recall_weighted']]

b1 = ax.bar(x - width/2, lstm_vals, width, label='BiLSTM', color='steelblue', alpha=0.85)
b2 = ax.bar(x + width/2, bert_vals, width, label='DistilBERT', color='purple', alpha=0.85)

ax.set_xticks(x); ax.set_xticklabels(metrics_to_plot)
ax.set_ylim(0.5, 1.05); ax.set_ylabel('Score')
ax.legend(); ax.grid(axis='y', alpha=0.3)

for bars in [b1, b2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', fontsize=8)

plt.tight_layout()
save_fig('06_lstm_vs_bert_metrics')

  Saved → /content/drive/MyDrive/Dataset/results/dl_plots/06_lstm_vs_bert_metrics.png
